# 🎓 Student Performance Predictor
### Fundamentals of AI and ML — BYOP Capstone Project

**Goal:** Predict whether a student will **pass or fail** based on study habits, attendance, family background, and prior academic performance.

**Dataset:** UCI Student Performance Dataset  
**Models Used:** Logistic Regression, Decision Tree Classifier  
**Target Variable:** `pass` (1 = G3 ≥ 10, 0 = G3 < 10)

---

## 📦 Section 1: Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn: preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Scikit-learn: models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Scikit-learn: evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ All libraries imported successfully!')

## 📥 Section 2: Load the Dataset

> **How to get the data:**  
> 1. Go to: https://www.kaggle.com/datasets/larsen0966/student-performance-data-set  
> 2. Download `student-mat.csv` (Mathematics dataset)  
> 3. Place it in the same folder as this notebook  
> 
> Alternatively, the cell below will auto-download it if `kaggle` CLI is configured.

In [ ]:
import os

# Try to load the file; if missing, provide clear instructions
FILENAME = 'student-mat.csv'

if not os.path.exists(FILENAME):
    print('⚠️  File not found. Attempting to download via kaggle CLI...')
    os.system('kaggle datasets download -d larsen0966/student-performance-data-set --unzip')

if os.path.exists(FILENAME):
    # The file uses semicolon as separator
    df = pd.read_csv(FILENAME, sep=';')
    print(f'✅ Dataset loaded! Shape: {df.shape}')
    print(f'   Rows: {df.shape[0]} students | Columns: {df.shape[1]} features')
else:
    print('❌ Please download student-mat.csv manually from Kaggle and place it here.')

In [ ]:
# Quick preview of the data
df.head()

In [ ]:
# Column descriptions
print('Column names:')
print(df.columns.tolist())

## 🔍 Section 3: Exploratory Data Analysis (EDA)

Before building a model, we explore the data to understand patterns, distributions, and relationships.

In [ ]:
# Basic statistics
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum().sum(), 'total missing values')

In [ ]:
# Summary statistics for numeric columns
df.describe().round(2)

In [ ]:
# Create the target variable: pass (1) or fail (0)
# G3 is the final grade (0–20); passing threshold = 10
df['pass'] = (df['G3'] >= 10).astype(int)

# Check class balance
pass_counts = df['pass'].value_counts()
print('Pass/Fail Distribution:')
print(f'  Pass (1): {pass_counts[1]} students ({pass_counts[1]/len(df)*100:.1f}%)')
print(f'  Fail (0): {pass_counts[0]} students ({pass_counts[0]/len(df)*100:.1f}%)')

# Plot class distribution
fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#e74c3c', '#2ecc71']
ax.bar(['Fail (G3 < 10)', 'Pass (G3 ≥ 10)'], pass_counts.sort_index(), color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('Student Pass/Fail Distribution', fontweight='bold')
ax.set_ylabel('Number of Students')
for i, v in enumerate(pass_counts.sort_index()):
    ax.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_01_class_distribution.png')

In [ ]:
# Distribution of final grades (G3)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['G3'], bins=21, range=(-0.5, 20.5), color='#3498db', edgecolor='white', linewidth=0.8)
ax.axvline(x=9.5, color='red', linestyle='--', linewidth=2, label='Pass threshold (10)')
ax.set_title('Distribution of Final Grades (G3)', fontweight='bold')
ax.set_xlabel('Final Grade (G3)')
ax.set_ylabel('Number of Students')
ax.legend()
ax.set_xticks(range(0, 21))
plt.tight_layout()
plt.savefig('plot_02_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_02_grade_distribution.png')

In [ ]:
# Study time vs pass rate
study_labels = {1: '<2 hrs', 2: '2–5 hrs', 3: '5–10 hrs', 4: '>10 hrs'}
df['studytime_label'] = df['studytime'].map(study_labels)
study_pass = df.groupby('studytime')['pass'].mean() * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    [study_labels[k] for k in study_pass.index],
    study_pass.values,
    color=['#e74c3c', '#e67e22', '#2ecc71', '#27ae60'],
    edgecolor='white', linewidth=1.5
)
ax.set_title('Study Time vs Pass Rate', fontweight='bold')
ax.set_xlabel('Weekly Study Time')
ax.set_ylabel('Pass Rate (%)')
ax.set_ylim(0, 100)
for bar, val in zip(bars, study_pass.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('plot_03_studytime_vs_passrate.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_03_studytime_vs_passrate.png')

In [ ]:
# Absences vs final grade (scatter)
fig, ax = plt.subplots(figsize=(10, 5))
colors_scatter = df['pass'].map({1: '#2ecc71', 0: '#e74c3c'})
ax.scatter(df['absences'], df['G3'], c=colors_scatter, alpha=0.6, edgecolors='white', linewidth=0.5)
ax.axhline(y=10, color='navy', linestyle='--', linewidth=1.5, label='Pass threshold')
ax.set_title('Absences vs Final Grade', fontweight='bold')
ax.set_xlabel('Number of Absences')
ax.set_ylabel('Final Grade (G3)')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Pass'), Patch(facecolor='#e74c3c', label='Fail')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('plot_04_absences_vs_grade.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_04_absences_vs_grade.png')

In [ ]:
# Correlation heatmap (numeric features only)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Focus on most relevant numeric features
key_numeric = ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
               'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health',
               'absences', 'G1', 'G2', 'G3', 'pass']

fig, ax = plt.subplots(figsize=(14, 10))
corr = df[key_numeric].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Feature Correlation Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plot_05_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_05_correlation_heatmap.png')
print('\nTop correlations with pass/fail:')
print(corr['pass'].drop('pass').sort_values(ascending=False).head(10).round(3))

## ⚙️ Section 4: Data Preprocessing

We prepare the data for machine learning:
1. Encode categorical features (text → numbers)
2. Drop the raw grade columns (G1, G2, G3) to make a **realistic** predictor
3. Split into training and test sets

In [ ]:
# Drop helper columns we added for EDA
df_model = df.drop(columns=['studytime_label'], errors='ignore')

# Drop G1, G2, G3 — we don't want to 'cheat' by using existing grades
# The model should predict based on background & habits, not prior grades
df_model = df_model.drop(columns=['G1', 'G2', 'G3'])

print('Features used for prediction:')
print([col for col in df_model.columns if col != 'pass'])

In [ ]:
# Identify categorical (text) columns
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns to encode ({len(cat_cols)}): {cat_cols}')

# One-hot encode categorical features
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
print(f'\nShape before encoding: {df_model.shape}')
print(f'Shape after encoding:  {df_encoded.shape}')

In [ ]:
# Separate features (X) and target (y)
X = df_encoded.drop(columns=['pass'])
y = df_encoded['pass']

print(f'Feature matrix X: {X.shape}')
print(f'Target vector  y: {y.shape}')
print(f'\nClass balance in y:')
print(y.value_counts())

In [ ]:
# Train-test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:   {X_train.shape[0]} samples')
print(f'Test set:       {X_test.shape[0]} samples')
print(f'\nPass rate in train: {y_train.mean()*100:.1f}%')
print(f'Pass rate in test:  {y_test.mean()*100:.1f}%')

## 🤖 Section 5: Model Training

We train **two models** and compare them:
- **Logistic Regression** — a linear model, interpretable, great baseline
- **Decision Tree** — a non-linear model, visually explainable

In [ ]:
# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('=== Logistic Regression ===')
print(f'Training Accuracy: {lr_model.score(X_train, y_train)*100:.2f}%')
print(f'Test Accuracy:     {accuracy_score(y_test, y_pred_lr)*100:.2f}%')

In [ ]:
# --- Model 2: Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print('=== Decision Tree ===')
print(f'Training Accuracy: {dt_model.score(X_train, y_train)*100:.2f}%')
print(f'Test Accuracy:     {accuracy_score(y_test, y_pred_dt)*100:.2f}%')

## 📊 Section 6: Model Evaluation

We evaluate both models using:
- **Accuracy** — overall correct predictions
- **Precision / Recall / F1** — especially important for imbalanced classes
- **Confusion Matrix** — visualise what the model got right and wrong

In [ ]:
# Detailed classification report — Logistic Regression
print('=== Classification Report: Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Fail', 'Pass']))

In [ ]:
# Detailed classification report — Decision Tree
print('=== Classification Report: Decision Tree ===')
print(classification_report(y_test, y_pred_dt, target_names=['Fail', 'Pass']))

In [ ]:
# Confusion Matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models = [
    (y_pred_lr, 'Logistic Regression', axes[0]),
    (y_pred_dt, 'Decision Tree',       axes[1])
]

for y_pred, title, ax in models:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_06_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_06_confusion_matrices.png')

In [ ]:
# Model comparison bar chart
metrics = ['Accuracy', 'Precision (Pass)', 'Recall (Pass)', 'F1 (Pass)']

from sklearn.metrics import precision_score, recall_score, f1_score

lr_scores = [
    accuracy_score(y_test, y_pred_lr),
    precision_score(y_test, y_pred_lr),
    recall_score(y_test, y_pred_lr),
    f1_score(y_test, y_pred_lr)
]
dt_scores = [
    accuracy_score(y_test, y_pred_dt),
    precision_score(y_test, y_pred_dt),
    recall_score(y_test, y_pred_dt),
    f1_score(y_test, y_pred_dt)
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars1 = ax.bar(x - width/2, lr_scores, width, label='Logistic Regression', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width/2, dt_scores, width, label='Decision Tree',       color='#e67e22', edgecolor='white')

ax.set_title('Model Comparison: Logistic Regression vs Decision Tree', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.legend()
ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('plot_07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_07_model_comparison.png')

## 💡 Section 7: Feature Importance

Which features matter most for predicting student performance?

In [ ]:
# Decision Tree feature importance
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Show top 15 features
top15 = importance_df.head(15)

fig, ax = plt.subplots(figsize=(11, 7))
colors = ['#e74c3c' if imp > top15['Importance'].mean() else '#3498db'
          for imp in top15['Importance']]
ax.barh(top15['Feature'][::-1], top15['Importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Top 15 Most Important Features (Decision Tree)', fontweight='bold', fontsize=13)
ax.set_xlabel('Importance Score')
ax.axvline(x=top15['Importance'].mean(), color='gray', linestyle='--', linewidth=1.5, label='Mean importance')
ax.legend()
plt.tight_layout()
plt.savefig('plot_08_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_08_feature_importance.png')
print('\nTop 10 features:')
print(importance_df.head(10).to_string(index=False))

In [ ]:
# Logistic Regression coefficients (top positive and negative predictors)
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)

top_pos = coef_df.head(8)
top_neg = coef_df.tail(8)
top_coef = pd.concat([top_pos, top_neg])

fig, ax = plt.subplots(figsize=(11, 7))
bar_colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in top_coef['Coefficient']]
ax.barh(top_coef['Feature'], top_coef['Coefficient'], color=bar_colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=1)
ax.set_title('Logistic Regression Coefficients\n(Green = increases pass probability, Red = decreases)', fontweight='bold', fontsize=12)
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.savefig('plot_09_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_09_lr_coefficients.png')

In [ ]:
# Visualise the Decision Tree (top 3 levels)
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=['Fail', 'Pass'],
    max_depth=3,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
ax.set_title('Decision Tree Visualisation (Max Depth = 3)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('plot_10_decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot saved: plot_10_decision_tree.png')

## 🔮 Section 8: Make a Prediction

Let's use our model to predict whether a **hypothetical student** will pass or fail.

In [ ]:
def predict_student(study_time=2, failures=0, absences=5, goout=2,
                    health=3, Walc=1, Dalc=1, freetime=3,
                    model_choice='decision_tree'):
    """
    Predict pass/fail for a student.
    
    Parameters:
    -----------
    study_time : int, 1-4 (1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs)
    failures   : int, 0-4 (number of past class failures)
    absences   : int, 0-93 (number of school absences)
    goout      : int, 1-5 (going out with friends; 1=very low, 5=very high)
    health     : int, 1-5 (current health status; 1=very bad, 5=very good)
    Walc       : int, 1-5 (weekend alcohol consumption)
    Dalc       : int, 1-5 (workday alcohol consumption)
    freetime   : int, 1-5 (free time after school)
    model_choice : 'logistic_regression' or 'decision_tree'
    """
    # Create a sample row matching training data structure
    sample = pd.DataFrame(np.zeros((1, X.shape[1])), columns=X.columns)
    
    # Set the known features
    for col in ['studytime', 'failures', 'absences', 'goout', 'health', 'Walc', 'Dalc', 'freetime']:
        if col in sample.columns:
            sample[col] = locals()[col if col != 'study_time' else 'study_time']
    
    # Fix studytime key
    if 'studytime' in sample.columns:
        sample['studytime'] = study_time
    
    # Choose model
    model = dt_model if model_choice == 'decision_tree' else lr_model
    model_name = 'Decision Tree' if model_choice == 'decision_tree' else 'Logistic Regression'
    
    prediction = model.predict(sample)[0]
    probability = model.predict_proba(sample)[0]
    
    result = '✅ PASS' if prediction == 1 else '❌ FAIL'
    print(f'=== Student Prediction ({model_name}) ===')
    print(f'Study time: {study_labels[study_time]}')
    print(f'Past failures: {failures}')
    print(f'Absences: {absences}')
    print(f'Social activity (goout): {goout}/5')
    print(f'\nPrediction: {result}')
    print(f'Confidence: Fail={probability[0]*100:.1f}%  Pass={probability[1]*100:.1f}%')
    return prediction, probability

# --- Example 1: Diligent student ---
print('--- Student A: Studies a lot, rarely absent ---')
predict_student(study_time=3, failures=0, absences=2, goout=1)

print()

# --- Example 2: Struggling student ---
print('--- Student B: Studies little, many absences, past failures ---')
predict_student(study_time=1, failures=2, absences=20, goout=5)

## 📝 Section 9: Results Summary

In [ ]:
# Final summary table
from sklearn.metrics import precision_score, recall_score, f1_score

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Train Accuracy': [
        f"{lr_model.score(X_train, y_train)*100:.2f}%",
        f"{dt_model.score(X_train, y_train)*100:.2f}%"
    ],
    'Test Accuracy': [
        f"{accuracy_score(y_test, y_pred_lr)*100:.2f}%",
        f"{accuracy_score(y_test, y_pred_dt)*100:.2f}%"
    ],
    'F1 Score (Pass)': [
        f"{f1_score(y_test, y_pred_lr):.3f}",
        f"{f1_score(y_test, y_pred_dt):.3f}"
    ],
    'Precision': [
        f"{precision_score(y_test, y_pred_lr):.3f}",
        f"{precision_score(y_test, y_pred_dt):.3f}"
    ],
    'Recall': [
        f"{recall_score(y_test, y_pred_lr):.3f}",
        f"{recall_score(y_test, y_pred_dt):.3f}"
    ]
})

print('=== FINAL RESULTS SUMMARY ===')
print(summary.to_string(index=False))
print()
print('Key findings:')
print('• Past failures is the strongest predictor of current performance')
print('• Study time positively correlates with passing')
print('• High absences strongly associated with failing')
print('• Both models perform reasonably well without using prior grade data')

---

## ✅ Project Complete!

### What we built:
- Loaded and explored the UCI Student Performance dataset
- Created a binary classification target (`pass`/`fail`)
- Trained and evaluated **Logistic Regression** and **Decision Tree** models
- Identified the most important features driving student performance
- Built a reusable prediction function

### Generated plots (save these for your report):
| File | Description |
|------|-------------|
| `plot_01_class_distribution.png` | Pass/Fail balance |
| `plot_02_grade_distribution.png` | G3 grade histogram |
| `plot_03_studytime_vs_passrate.png` | Study habits vs outcome |
| `plot_04_absences_vs_grade.png` | Attendance vs performance |
| `plot_05_correlation_heatmap.png` | Feature correlations |
| `plot_06_confusion_matrices.png` | Model prediction errors |
| `plot_07_model_comparison.png` | Side-by-side model scores |
| `plot_08_feature_importance.png` | Decision Tree feature ranking |
| `plot_09_lr_coefficients.png` | Logistic Regression weights |
| `plot_10_decision_tree.png` | Decision Tree visualisation |

### Possible extensions (for bonus marks):
- Add **Random Forest** as a third model
- Try **cross-validation** instead of a single train/test split
- Experiment with including G1 and G2 — how much does accuracy improve?
- Build a simple interactive prediction form using `ipywidgets`